# The infrared divergence of double Compton scattering cancels

Chluba's thesis (section 4.4.5) stops the infrared divergence of double Compton emission with a lowest
frequency. In dimensional regularisation ($D = 4 - 2\epsilon$) the same divergence is a pole $1/\epsilon$ and it
cancels against the one-loop virtual correction (Bloch–Nordsieck). The soft-photon factor depends only on
the charged legs (the electron, $P \to P'$), so its virtual partner is the infrared pole of the electron
vertex at $t = (P - P')^2$:
- real, one soft photon with energy below $\Delta E$: $d\sigma_C\,\big(-\alpha I(t)/(2\pi\epsilon) + \text{finite}\big)$
- virtual: $d\sigma_C\cdot 2\,\mathrm{Re}\,F_1(t)$

with $I(t) = 2P\cdot P'\int_0^1 \frac{dx}{m^2 - x(1 - x)t} - 2$, the angular integral of the eikonal factor.
So the infrared pole of $F_1$ must be $+\alpha I(t)/(4\pi\epsilon)$. The ultraviolet pole of $F_1$ does not depend on $t$,
so the difference between two values of $t$ shows the infrared part alone.

The same calculation with FeynCalc and Package-X is in `ir_cancellation_feyncalc.wl`. Run this notebook with
the SageMath kernel, from this folder or from the top folder of the repository.

In [1]:
import sys, os
# find the repository (this notebook may be opened from examples/chluba or from the top folder)
root = os.getcwd()
while not os.path.isfile(os.path.join(root, "feynsage", "__init__.py")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, root)
sys.path.insert(0, os.path.join(root, "examples", "chluba"))
from feynsage import *
from feynsage.dirac import projector, form_factor

## 1. The pole of $F_1(t)$ against $I(t)$

`projector('F1', ...)` with `form_factor` gives $F_1$ of the one-loop vertex directly, in units $\alpha/4\pi$
(see `examples/feynsage_first_steps.ipynb`, Section 7, for every argument). The projector needs the same
$q^2 = t$ as the kinematics. We set $m = 1$ and $\mu = 1$.

In [2]:
m = var('m')
line = ['rho', 'p2 - l + m', 'mu', 'p1 - l + m', 'rho']           # the electron line of the vertex
pole = {}
for t in [-1/2, -1, -3, -8]:
    F1 = form_factor(line, projector('F1', 'mu', 'p1', 'm', 'p2', 'm', q2=t),
                     ['l', '0'], ['l - p2', 'm'], ['l - p1', 'm'],
                     kin={'p1^2': 'm^2', 'p2^2': 'm^2', 'p1.p2': 'm^2 - (%s)/2' % t}).coefficients()['1']
    pole[t] = uv_part(F1).subs(m=1, mu=1)                          # coefficient of 1/eps
x = var('x')
I = lambda t: 2*(1 - t/2)*numerical_integral(1/(1 - x*(1 - x)*t), 0, 1)[0] - 2
for t in [-1, -3, -8]:
    print("t = %3s:  pole of F1(t) - F1(-1/2) = %.12f    I(t) - I(-1/2) = %.12f"
          % (t, (pole[t] - pole[-1/2]).n().real(), I(t) - I(-1/2)))

t =  -1:  pole of F1(t) - F1(-1/2) = 0.271963043918    I(t) - I(-1/2) = 0.271963043918
t =  -3:  pole of F1(t) - F1(-1/2) = 1.108545637288    I(t) - I(-1/2) = 1.108545637288
t =  -8:  pole of F1(t) - F1(-1/2) = 2.368915948651    I(t) - I(-1/2) = 2.368915948651


The real and the virtual poles cancel. With a photon mass, or in dimensional regularisation, the lowest
frequency of the thesis becomes a logarithm of the energy resolution $\Delta E$.

**A check of the normalisation.** The same vertex with the projector for $F_2$ must give the electron's
$g - 2$: $F_2(0) = \frac{\alpha}{4\pi}\times 2 = \frac{\alpha}{2\pi}$.

In [3]:
q2 = var('q2')
F2 = form_factor(line, projector('F2', 'mu', 'p1', 'm', 'p2', 'm'),
                 ['l', '0'], ['l - p2', 'm'], ['l - p1', 'm'],
                 kin={'p1^2': 'm^2', 'p2^2': 'm^2', 'p1.p2': 'm^2 - q2/2'}).coefficients()['1']
print("1/eps pole of F2:", uv_part(F2))
print("F2 at q^2 = 0, in units alpha/(4 pi):", finite_part(F2, 1).subs(q2=0, m=1).n(digits=12).real())

1/eps pole of F2: 0
F2 at q^2 = 0, in units alpha/(4 pi): 2.00000000000


## 2. Lightman's law for soft photon emission

One soft photon per Compton scattering comes with $dN = \frac{\alpha}{\pi}\, I(t)\,\frac{d\omega_2}{\omega_2}$. For cold
electrons and soft incoming photons $-t = 2\omega_0^2(1 - \cos\theta)$ is small. Averaged over the Thomson
distribution $1 + \cos^2\theta$ this gives Lightman's law (thesis eq. 4.24), $dN/d\omega_2 = \frac{4\alpha}{3\pi}\,\omega_0^2/\omega_2$
(with $m = 1$):

In [4]:
t_, c_, w0 = var('t_ c_ w0')
I_small_t = (2*(1 - t_/2)*integrate((1/(1 - x*(1 - x)*t_)).series(t_, 3).truncate(), x, 0, 1) - 2).series(t_, 2).truncate()
thomson = 1 + c_^2
average = integrate(I_small_t.subs(t_=-2*w0^2*(1 - c_))*thomson, c_, -1, 1) / integrate(thomson, c_, -1, 1)
print("I(t) for small t   :", I_small_t)
print("Thomson average    :", average.series(w0, 3).truncate())

I(t) for small t   : -2/3*t_
Thomson average    : 4/3*w0^2
